In [3]:
import torch
import sys
import shutil

print("Python :", sys.version)
print("PyTorch :", torch.__version__)
print("CUDA PyTorch :", torch.version.cuda)
print("GPU disponible :", torch.cuda.is_available())
print("Compilateur nvcc :", shutil.which("nvcc"))

assert torch.cuda.is_available(), "Active le GPU dans Colab."
assert shutil.which("nvcc"), "Compilateur CUDA introuvable."

Python : 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch : 2.14.1+cu130
CUDA PyTorch : 13.0
GPU disponible : True
Compilateur nvcc : /usr/local/cuda/bin/nvcc


In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
from pathlib import Path

backup = Path("/content/drive/MyDrive/data/images_tasse/colmap_light")

print("Sauvegarde :", backup.exists())
print("Images :", (backup / "images").exists())
print("Reconstruction :", (backup / "sparse/0/cameras.bin").exists())

Sauvegarde : True
Images : True
Reconstruction : True


In [6]:
import shutil
from pathlib import Path

source = Path("/content/drive/MyDrive/data/images_tasse/colmap_light")
destination = Path("/content/tasse_light")

assert (source / "images").is_dir()
assert (source / "sparse/0/cameras.bin").exists()
assert (source / "sparse/0/images.bin").exists()
assert (source / "sparse/0/points3D.bin").exists()

shutil.copytree(source / "images", destination / "images", dirs_exist_ok=True)
shutil.copytree(source / "sparse", destination / "sparse", dirs_exist_ok=True)

print("Nombre de photos :", len(list((destination / "images").glob("*.jpeg"))))
print("Données COLMAP restaurées")

Nombre de photos : 67
Données COLMAP restaurées


In [7]:
%cd /content

!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive

%cd /content/gaussian-splatting

/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 1053, done.
remote: Total 1053 (delta 0), reused 0 (delta 0), pack-reused 1053 (from 1)
Receiving objects: 100% (1053/1053), 78.72 MiB | 36.96 MiB/s, done.
Resolving deltas: 100% (593/593), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects:

In [8]:
!python -m pip install plyfile tqdm opencv-python joblib

In [14]:
import os
os.environ["NVCC_PREPEND_FLAGS"] = "--pre-include cstdint"

%cd /content/gaussian-splatting

!python -m pip install --no-build-isolation ./submodules/diff-gaussian-rasterization

/content/gaussian-splatting
Processing ./submodules/diff-gaussian-rasterization
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp313-cp313-linux_x86_64.whl size=3643910 sha256=89f91964d5fa23ec590a3fef3bf56cbe6bcff0634d00651816179f8ce75c2d6a
  Stored in directory: /tmp/pip-ephem-wheel-cache-bsxm6q_9/wheels/49/ab/50/4f594db59dd034cb4fc9a6b5aae6986c5749bdd12e71eb12f0
Successfully built diff_gaussian_rasterization


In [15]:
!python -m pip install --no-build-isolation ./submodules/simple-knn

Processing ./submodules/simple-knn
  Preparing metadata (pyproject.toml) ... done
  Created wheel for simple_knn: filename=simple_knn-0.0.0-cp313-cp313-linux_x86_64.whl size=3493899 sha256=740f3ea6595af2370f310340e19bdfcd6d51d2435d1d0abf2c3e36dd34429562
  Stored in directory: /tmp/pip-ephem-wheel-cache-t67x5zwh/wheels/9c/6f/a6/86f8f308d51a82157b32336055b058903d71f74c2ca4e9cd85
Successfully built simple_knn


In [17]:
import torch
import diff_gaussian_rasterization
import simple_knn

print("GPU :", torch.cuda.get_device_name(0))
print("Extensions CUDA importées avec succès")

GPU : Tesla T4
Extensions CUDA importées avec succès


In [18]:
%cd /content/gaussian-splatting

!python train.py \
    -s /content/tasse_light \
    -m /content/drive/MyDrive/data/images_tasse/gaussian_model_3000 \
    --iterations 3000 \
    --disable_viewer

/content/gaussian-splatting
I0000 00:00:1794603769.063526    8875 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794603771.543251    8875 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Optimizing /content/drive/MyDrive/data/images_tasse/gaussian_model_3000
Output folder: /content/drive/MyDrive/data/images_tasse/gaussian_model_3000 [13/11 21:02:54]
Reading camera 67/67 [13/11 21:02:54]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [13/11 21:02:54]
Loading Training Cameras [13/11 21:02:55]
Loading Test Cameras [13/11 21:02:58]
Number of points at initialisation :  35410 [13/11 21:02:58]
Training progress: 100% 3000/3000 [04:43<00:00, 10.57it/s, Loss=0.0396821, Depth Loss=0.0000000]

[I

In [19]:
from pathlib import Path

model = Path("/content/drive/MyDrive/data/images_tasse/gaussian_model_3000/point_cloud/iteration_3000/point_cloud.ply")

print("Modèle enregistré :", model.exists())
print("Taille :", round(model.stat().st_size / 1e6, 2), "Mo" if model.exists() else "")

Modèle enregistré : True
Taille : 113.07 Mo
